# Customer Segmentation & Lifetime Value Analysis
### Who are our most valuable customers, and what should we do differently for each group?

---

**Business context:** Not all customers are equal. A small proportion of customers typically generate the majority of revenue, while a large portion buy once and never return. Understanding *who* your customers are — and treating them accordingly — is one of the highest-leverage things a data team can do for a business.

**Objective:** Segment Olist's 90,000+ customers by purchasing behaviour using the RFM framework (Recency, Frequency, Monetary value), estimate Customer Lifetime Value per segment, and translate findings into targeted business recommendations.

**Dataset:** Olist Brazilian E-Commerce dataset — real transaction data from 2016–2018 across 8 relational tables covering orders, customers, products, sellers, payments, and reviews.

**Approach:**
1. Explore the data and understand revenue and order trends
2. Engineer RFM features for every customer
3. Cluster customers into behavioural segments using K-Means
4. Profile each segment and estimate CLV
5. Recommend targeted actions per segment

## 1. Setup & Data Loading

The Olist dataset is structured as a relational database across 8 CSV files. Unlike a single flat file, we need to join these tables to build a complete picture of each customer's journey. Understanding the schema before writing any code is essential.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
path = "Data/Brazilian e-commerce/"

orders               = pd.read_csv(path + "olist_orders_dataset.csv")
customers            = pd.read_csv(path + "olist_customers_dataset.csv")
order_items          = pd.read_csv(path + "olist_order_items_dataset.csv")
products             = pd.read_csv(path + "olist_products_dataset.csv")
sellers              = pd.read_csv(path + "olist_sellers_dataset.csv")
payments             = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews              = pd.read_csv(path + "olist_order_reviews_dataset.csv")
geo                  = pd.read_csv(path + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")

print("Tables loaded successfully")
print(f"  orders:       {orders.shape}")
print(f"  customers:    {customers.shape}")
print(f"  order_items:  {order_items.shape}")
print(f"  products:     {products.shape}")
print(f"  sellers:      {sellers.shape}")
print(f"  payments:     {payments.shape}")
print(f"  reviews:      {reviews.shape}")
print(f"  geo:          {geo.shape}")

### Schema overview

The 8 tables connect like this:
```
customers ──── orders ──── order_items ──── products
                  │               └──────── sellers
                  ├──── payments
                  └──── reviews
```

The central table is **orders** — everything joins through it. Each order belongs to one customer, can have multiple items, multiple payments, and one review.

For our RFM analysis, we primarily need:
- `customers` → unique customer IDs
- `orders` → order dates and status
- `order_items` → item prices (revenue)
- `payments` → payment values (alternative revenue source)

In [ ]:
# Check join keys and date columns
print("=== orders ===")
print(orders.dtypes[["customer_id", "order_purchase_timestamp", "order_status"]])

print("\n=== customers ===")
print(customers.dtypes[["customer_id", "customer_unique_id", "customer_state"]])

print("\n=== order_items ===")
print(order_items.dtypes[["order_id", "price", "freight_value"]])

print("\n=== payments ===")
print(payments.dtypes[["order_id", "payment_value"]])

## 2. Exploratory Data Analysis

Before building the segmentation model, we explore the data to understand the business context — how orders and revenue have grown over time, which product categories drive the most sales, and where customers are located. This sets up the story we'll tell with the segmentation.

In [8]:
# Convert timestamps to datetime
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

# For RFM analysis we only want delivered orders — cancelled/unavailable orders
# don't represent real customer behaviour
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

print(f"Total orders:     {len(orders):,}")
print(f"Delivered orders: {len(orders_delivered):,}")
print(f"Date range:       {orders_delivered['order_purchase_timestamp'].min().date()} "
      f"to {orders_delivered['order_purchase_timestamp'].max().date()}")

Total orders:     99,441
Delivered orders: 96,478
Date range:       2016-09-15 to 2018-08-29


In [9]:
# Monthly order volume
orders_delivered["month"] = orders_delivered["order_purchase_timestamp"].dt.to_period("M")
monthly_orders = orders_delivered.groupby("month").size().reset_index(name="order_count")
monthly_orders["month"] = monthly_orders["month"].astype(str)

fig = px.line(
    monthly_orders,
    x="month", y="order_count",
    title="Monthly Order Volume (Delivered Orders)",
    labels={"month": "Month", "order_count": "Number of Orders"},
    markers=True
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [10]:
# Join orders with payments to get revenue
order_revenue = orders_delivered.merge(
    payments.groupby("order_id")["payment_value"].sum().reset_index(),
    on="order_id", how="left"
)

monthly_revenue = order_revenue.groupby("month")["payment_value"].sum().reset_index()
monthly_revenue["month"] = monthly_revenue["month"].astype(str)

fig = px.bar(
    monthly_revenue,
    x="month", y="payment_value",
    title="Monthly Revenue (R$)",
    labels={"month": "Month", "payment_value": "Revenue (R$)"},
    color="payment_value",
    color_continuous_scale="Blues"
)
fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
fig.show()

In [11]:
# Join order_items → products → category translation
items_with_category = order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
items_with_category = items_with_category.merge(category_translation, on="product_category_name", how="left")

top_categories = (
    items_with_category.groupby("product_category_name_english")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_categories,
    x="price", y="product_category_name_english",
    orientation="h",
    title="Top 10 Product Categories by Revenue (R$)",
    labels={"price": "Total Revenue (R$)", "product_category_name_english": "Category"}
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [12]:
orders_by_state = (
    orders_delivered.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
    .groupby("customer_state")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="order_count")
)

fig = px.bar(
    orders_by_state,
    x="customer_state", y="order_count",
    title="Orders by State",
    labels={"customer_state": "State", "order_count": "Number of Orders"},
    color="order_count",
    color_continuous_scale="Blues"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

### EDA Summary

- **96,478 delivered orders** across nearly 2 years (Sep 2016 – Aug 2018)
- **Strong growth trajectory** — the platform scaled from ~200 orders/month in late 2016 to ~7,000/month by late 2017, then stabilised. Revenue reached ~R$1M/month and held steady.
- **Health & beauty is the top category** by revenue, followed by watches/gifts and bed/bath/table — lifestyle and personal care products dominate
- **São Paulo dominates geographically** with ~40k orders — nearly 3× the next largest state (RJ). The business is heavily concentrated in the Southeast.

These patterns will become meaningful when we overlay them with customer segments — for example, do high-value customers concentrate in SP, or are they distributed differently?

## 3. RFM Feature Engineering

RFM is a proven framework for understanding customer behaviour based on three dimensions:

- **Recency (R)** — How recently did the customer last purchase? A customer who bought last week is more engaged than one who bought 18 months ago.
- **Frequency (F)** — How many times has the customer purchased? Repeat buyers are more loyal and more valuable.
- **Monetary (M)** — How much has the customer spent in total? High spenders contribute disproportionately to revenue.

By combining these three dimensions, we can profile every customer with a single row of data — and then cluster similar customers together into meaningful segments.

**Implementation note:** We use `customer_unique_id` (not `customer_id`) to identify customers. In the Olist dataset, a single customer can have multiple `customer_id` values across different orders, but only one `customer_unique_id`. Using the wrong column would cause us to treat the same person as multiple customers.

In [13]:
# Step 1: link orders to unique customer IDs
orders_with_uid = orders_delivered.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id", how="left"
)

# Step 2: link orders to payment values
orders_with_revenue = orders_with_uid.merge(
    payments.groupby("order_id")["payment_value"].sum().reset_index(),
    on="order_id", how="left"
)

# Step 3: compute RFM — one row per unique customer
snapshot_date = orders_delivered["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = (
    orders_with_revenue
    .groupby("customer_unique_id")
    .agg(
        recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("payment_value", "sum")
    )
    .reset_index()
)

print(f"RFM table shape: {rfm.shape}")
print(f"\nSnapshot date (reference point for recency): {snapshot_date.date()}")
print(f"\nSample:")
rfm.head()

RFM table shape: (93358, 4)

Snapshot date (reference point for recency): 2018-08-30

Sample:


,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
2,0000f46a3911fa3c0805444483337064,537,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [14]:
print("RFM Summary Statistics")
print("=" * 40)
print(rfm[["recency", "frequency", "monetary"]].describe().round(2))

RFM Summary Statistics
       recency  frequency  monetary
count 93358.00   93358.00  93358.00
mean    237.94       1.03    165.20
std     152.59       0.21    226.31
min       1.00       1.00      0.00
25%     114.00       1.00     63.05
50%     219.00       1.00    107.78
75%     346.00       1.00    182.56
max     714.00      15.00  13664.08


### Understanding the RFM distributions

Before clustering, it's important to understand the shape of each RFM dimension — outliers and skewed distributions can distort K-Means clustering, which is sensitive to scale.

In [15]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["Recency (days)", "Frequency (orders)", "Monetary (R$)"])

fig.add_trace(go.Histogram(x=rfm["recency"], nbinsx=50, marker_color="#636EFA", name="Recency"), row=1, col=1)
fig.add_trace(go.Histogram(x=rfm["frequency"], nbinsx=30, marker_color="#EF553B", name="Frequency"), row=1, col=2)
fig.add_trace(go.Histogram(x=rfm["monetary"], nbinsx=50, marker_color="#00CC96", name="Monetary"), row=1, col=3)

fig.update_layout(title="RFM Feature Distributions", showlegend=False, height=400)
fig.show()

print(f"\nCustomers with only 1 order: {(rfm['frequency'] == 1).sum():,} ({(rfm['frequency'] == 1).mean()*100:.1f}%)")
print(f"Customers with 2+ orders:    {(rfm['frequency'] > 1).sum():,} ({(rfm['frequency'] > 1).mean()*100:.1f}%)")


Customers with only 1 order: 90,557 (97.0%)
Customers with 2+ orders:    2,801 (3.0%)


### Key observations from the distributions

- **Recency is spread across the full range** — some customers bought very recently, others over a year ago
- **Frequency is heavily right-skewed** — the vast majority of customers placed only 1 order. This is typical for e-commerce platforms and makes frequency a critical differentiator between casual and loyal customers.
- **Monetary is also right-skewed** — most customers have modest spend, but a long tail of high-value customers exists

Because of the skew in frequency and monetary, we apply **log transformation** before clustering to prevent high-spend outliers from dominating the model.

In [16]:
from sklearn.preprocessing import StandardScaler
import numpy as np

rfm_model = rfm.copy()

# Log-transform frequency and monetary to reduce skew
# Adding 1 avoids log(0) errors
rfm_model["frequency_log"] = np.log1p(rfm_model["frequency"])
rfm_model["monetary_log"] = np.log1p(rfm_model["monetary"])

# Standardise all three features so they're on the same scale for K-Means
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_model[["recency", "frequency_log", "monetary_log"]])

print("Scaling complete.")
print(f"Shape of scaled RFM matrix: {rfm_scaled.shape}")

Scaling complete.
Shape of scaled RFM matrix: (93358, 3)


### Key observations from the distributions

- **Recency is spread across the full range** — customers are distributed across the entire 2-year window, with a slight concentration around 200–300 days ago
- **97% of customers placed only 1 order** — just 2,801 customers (3%) ever returned for a second purchase. This is Olist's most critical business problem: almost no one comes back. Frequency will be the sharpest differentiator between our segments.
- **Monetary is heavily right-skewed** — most customers spent R$63–183 (25th–75th percentile), but a long tail extends to R$13,664. High-value customers are rare but disproportionately important.

Because of the skew in frequency and monetary, we apply **log transformation** before clustering to prevent high-spend outliers from dominating the model.

## 4. Customer Segmentation — K-Means Clustering

With our RFM features prepared, we now cluster customers into segments. We use **K-Means**, which groups customers by minimising the distance between each customer and their cluster's centre.

The key question is: **how many clusters?** Too few and we lose meaningful distinctions. Too many and segments become hard to act on. We use the **Elbow Method** and **Silhouette Score** to find the optimal number.

In [17]:
inertias = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, kmeans.labels_))
    print(f"k={k}  inertia={kmeans.inertia_:.0f}  silhouette={silhouette_scores[-1]:.3f}")

k=2  inertia=190197  silhouette=0.708
k=3  inertia=126273  silhouette=0.358
k=4  inertia=85782  silhouette=0.369
k=5  inertia=70389  silhouette=0.354
k=6  inertia=59335  silhouette=0.344
k=7  inertia=50741  silhouette=0.358
k=8  inertia=45168  silhouette=0.348


In [18]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Elbow Method (Inertia)", "Silhouette Score"])

fig.add_trace(go.Scatter(x=list(k_range), y=inertias,
                         mode="lines+markers", marker_color="#636EFA", name="Inertia"),
              row=1, col=1)

fig.add_trace(go.Scatter(x=list(k_range), y=silhouette_scores,
                         mode="lines+markers", marker_color="#EF553B", name="Silhouette"),
              row=1, col=2)

fig.update_layout(title="Choosing the Optimal Number of Clusters", height=400, showlegend=False)
fig.update_xaxes(title_text="Number of Clusters (k)")
fig.update_yaxes(title_text="Inertia", row=1, col=1)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)
fig.show()

### Choosing k = 4

The silhouette score peaks at k=2 (0.708) but drops sharply to ~0.36 for k≥3 and then plateaus — meaning there is no strong mathematical signal favouring any particular k in the 3–8 range. In cases like this, the right choice is driven by **business interpretability**.

- **k=2** — too coarse. Would simply split customers into "active" vs "inactive", which gives the business nothing actionable.
- **k=4** — the elbow shows a clear bend here, and four segments map naturally to recognisable customer archetypes: Champions, Loyal, At-Risk, and Lost. Each segment is distinct enough to warrant a different marketing or retention strategy.
- **k=5+** — diminishing returns. Segments start to overlap and become difficult to explain or act on.

We proceed with **k=4**.

In [19]:
# Fit final K-Means model with k=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_model["cluster"] = kmeans_final.fit_predict(rfm_scaled)

print("Cluster sizes:")
print(rfm_model["cluster"].value_counts().sort_index())

Cluster sizes:
cluster
0    35696
1    27816
2     2801
3    27045
Name: count, dtype: int64


In [20]:
# Profile each cluster to assign meaningful names
cluster_profile = rfm_model.groupby("cluster").agg(
    recency_mean=("recency", "mean"),
    frequency_mean=("frequency", "mean"),
    monetary_mean=("monetary", "mean"),
    customer_count=("customer_unique_id", "count")
).round(2)

cluster_profile["revenue_share"] = (
    cluster_profile["customer_count"] * cluster_profile["monetary_mean"] /
    (cluster_profile["customer_count"] * cluster_profile["monetary_mean"]).sum() * 100
).round(1)

print(cluster_profile.sort_values("monetary_mean", ascending=False))

         recency_mean  frequency_mean  monetary_mean  customer_count  \
cluster                                                                
1              174.11            1.00         318.84           27816   
2              220.29            2.11         308.59            2801   
3              425.09            1.00         119.01           27045   
0              147.27            1.00          69.22           35696   

         revenue_share  
cluster                 
1                57.50  
2                 5.60  
3                20.90  
0                16.00  


In [21]:
segment_map = {
    0: "Low-Value Inactive",
    1: "High-Value Inactive",
    2: "Loyal Customers",
    3: "Lost Customers"
}

rfm_model["segment"] = rfm_model["cluster"].map(segment_map)

print("Segment distribution:")
print(rfm_model["segment"].value_counts())

Segment distribution:
segment
Low-Value Inactive     35696
High-Value Inactive    27816
Lost Customers         27045
Loyal Customers         2801
Name: count, dtype: int64


### Segment profiles

| Segment | Customers | Avg Recency | Avg Frequency | Avg Spend | Revenue Share |
|---|---|---|---|---|---|
| High-Value Inactive | 27,816 | 174 days | 1.0 | R$319 | 57.5% |
| Low-Value Inactive | 35,696 | 147 days | 1.0 | R$69 | 16.0% |
| Lost Customers | 27,045 | 425 days | 1.0 | R$119 | 20.9% |
| Loyal Customers | 2,801 | 220 days | 2.1 | R$309 | 5.6% |

**Key observations:**
- **High-Value Inactive customers generate 57.5% of total revenue** despite being only 30% of the customer base — a textbook illustration of the 80/20 rule
- **97% of customers bought only once** — visible here as Loyal Customers being by far the smallest segment (2,801 people)
- **Lost Customers** last purchased 425 days ago on average — over 14 months. Reactivation is unlikely and expensive; resources are better spent elsewhere
- The difference between High-Value and Low-Value Inactive is purely spend (R$319 vs R$69) — both bought recently and only once. Spend level is likely driven by product category choice.

In [22]:
segment_summary = rfm_model.groupby("segment").agg(
    customer_count=("customer_unique_id", "count"),
    recency_mean=("recency", "mean"),
    frequency_mean=("frequency", "mean"),
    monetary_mean=("monetary", "mean")
).round(1).reset_index()

segment_summary["revenue_total"] = segment_summary["customer_count"] * segment_summary["monetary_mean"]
segment_summary["revenue_share"] = (segment_summary["revenue_total"] / segment_summary["revenue_total"].sum() * 100).round(1)

# Chart 1: customer count per segment
fig1 = px.bar(
    segment_summary.sort_values("customer_count", ascending=False),
    x="segment", y="customer_count",
    title="Customer Count by Segment",
    labels={"segment": "Segment", "customer_count": "Number of Customers"},
    color="segment",
    color_discrete_sequence=["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
)
fig1.update_layout(showlegend=False)
fig1.show()

# Chart 2: revenue share per segment
fig2 = px.pie(
    segment_summary,
    names="segment", values="revenue_share",
    title="Revenue Share by Segment (%)",
    color_discrete_sequence=["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
)
fig2.show()

# Chart 3: RFM scatter — recency vs monetary, coloured by segment
fig3 = px.scatter(
    rfm_model.sample(5000, random_state=42),
    x="recency", y="monetary",
    color="segment",
    title="Customer Segments — Recency vs Monetary Spend (sample of 5,000)",
    labels={"recency": "Recency (days since last order)", "monetary": "Total Spend (R$)"},
    opacity=0.6,
    color_discrete_sequence=["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
)
fig3.update_layout(height=500)
fig3.show()

## 5. Customer Lifetime Value Estimation

Segmenting customers tells us *who* they are. Estimating Customer Lifetime Value (CLV) tells us *how much they're worth* — giving the business a financial basis for deciding how much to invest in acquiring or retaining each segment.

We use a simple but practical CLV formula:

> **CLV = Average Order Value × Purchase Frequency × Customer Lifespan (years)**

Where:
- **Average Order Value** = total spend ÷ number of orders
- **Purchase Frequency** = orders per customer per year
- **Customer Lifespan** = assumed 3 years (a standard assumption for e-commerce in the absence of churn rate data)

This is a historical/descriptive CLV — it projects current behaviour forward, rather than predicting future behaviour probabilistically. For a more sophisticated model, BG/NBD or Pareto/NBD models would be used.

In [23]:
# Dataset spans ~2 years (Sep 2016 – Aug 2018)
dataset_years = 2.0
assumed_lifespan_years = 3.0

clv_summary = rfm_model.groupby("segment").agg(
    customer_count=("customer_unique_id", "count"),
    avg_order_value=("monetary", "mean"),
    avg_frequency=("frequency", "mean"),
    total_revenue=("monetary", "sum")
).round(2).reset_index()

# Annualise frequency (orders per year)
clv_summary["annual_frequency"] = (clv_summary["avg_frequency"] / dataset_years).round(2)

# CLV = AOV × annual frequency × lifespan
clv_summary["estimated_clv"] = (
    clv_summary["avg_order_value"] *
    clv_summary["annual_frequency"] *
    assumed_lifespan_years
).round(2)

clv_summary["revenue_share_pct"] = (
    clv_summary["total_revenue"] / clv_summary["total_revenue"].sum() * 100
).round(1)

print(clv_summary[["segment", "customer_count", "avg_order_value",
                    "annual_frequency", "estimated_clv", "revenue_share_pct"]]
      .sort_values("estimated_clv", ascending=False)
      .to_string(index=False))

            segment  customer_count  avg_order_value  annual_frequency  estimated_clv  revenue_share_pct
    Loyal Customers            2801           308.59              1.06         981.32               5.60
High-Value Inactive           27816           318.84              0.50         478.26              57.50
     Lost Customers           27045           119.01              0.50         178.52              20.90
 Low-Value Inactive           35696            69.22              0.50         103.83              16.00


In [24]:
clv_sorted = clv_summary.sort_values("estimated_clv", ascending=False)

fig = px.bar(
    clv_sorted,
    x="segment", y="estimated_clv",
    title="Estimated Customer Lifetime Value by Segment (R$)",
    labels={"segment": "Segment", "estimated_clv": "Estimated CLV (R$)"},
    color="segment",
    text="estimated_clv",
    color_discrete_sequence=["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
)
fig.update_traces(texttemplate="R$%{text:.0f}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Estimated CLV (R$)")
fig.show()

### CLV interpretation

The CLV estimates confirm and quantify what the segmentation already suggested:

- **Loyal Customers have the highest CLV** — their repeat purchasing behaviour, even at a modest rate of ~1 order/year, compounds over a 3-year lifespan. These customers should be treated as VIPs and protected at all costs.
- **High-Value Inactive customers represent the biggest opportunity** — they spent R$319 on average but only purchased once. If even 10% could be reactivated into repeat buyers, the revenue impact would be substantial given there are 27,816 of them.
- **Low-Value Inactive customers have low CLV** — their spend per order is modest (R$69) and they show no repeat behaviour. Marketing spend on this segment should be minimal.
- **Lost Customers should not be targeted for reactivation** — their last purchase was over 14 months ago and their CLV is low. Resources spent here have poor expected return.

## 6. Conclusions & Business Recommendations

### What we found

This analysis of 93,358 Olist customers reveals a business with strong acquisition but a critical retention problem:

1. **97% of customers never returned for a second purchase** — Olist is almost entirely dependent on new customer acquisition to sustain revenue. This is expensive and unsustainable.

2. **The top 30% of customers (High-Value Inactive) generate 57.5% of revenue** — a classic power law distribution. These customers are high-value but at risk of being lost forever if not re-engaged.

3. **Loyal Customers are tiny but disproportionately valuable** — just 2,801 customers (3%) who bought more than once represent the blueprint for what a healthy customer base should look like.

4. **Health & beauty, watches/gifts, and bed/bath dominate revenue** — product strategy and promotions should prioritise these categories.

5. **São Paulo drives ~40% of all orders** — geographic expansion to other major states (RJ, MG) represents a growth opportunity.

---

### Recommended actions by segment

| Segment | Size | Priority | Recommended action |
|---|---|---|---|
| **Loyal Customers** | 2,801 | 🔴 Highest | VIP programme — early access, exclusive discounts, personalised communication. Protect and nurture these customers above all others. |
| **High-Value Inactive** | 27,816 | 🔴 High | Win-back campaign — personalised email with discount on categories they previously bought. Target within 6 months of last purchase before they become truly lost. |
| **Low-Value Inactive** | 35,696 | 🟡 Medium | Low-cost re-engagement — automated email sequence, no heavy discounting. Focus on moving a subset into higher-value categories (e.g., health & beauty upsell). |
| **Lost Customers** | 27,045 | 🟢 Low | Suppress from marketing spend. Last purchase was 14+ months ago — reactivation cost exceeds expected return for most of this segment. |

---

### The single most important recommendation

**Olist should build a post-purchase nurture sequence targeting all first-time buyers within 30–60 days of their first order.** Given that 97% of customers never return, even a modest improvement in repeat purchase rate (e.g., from 3% to 6%) would double the size of the Loyal Customers segment and significantly increase long-term revenue without increasing acquisition spend.

---

### Limitations & next steps

- **CLV estimates are simplified** — a BG/NBD probabilistic model would give more accurate individual-level predictions
- **No product-level analysis** — combining segment data with product preferences would enable truly personalised recommendations
- **Geographic segmentation** — layering state-level data onto segments could reveal regional patterns worth acting on
- **Churn prediction model** — building a classifier to identify which High-Value Inactive customers are most likely to lapse would allow more precise targeting